In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange
import concurrent.futures
import unittest
from concurrent.futures import ThreadPoolExecutor
import PIL.Image
import cairosvg
from io import BytesIO

sys.path.append("../../")
import biked_commons
from biked_commons.api.rendering import RenderingEngine
from biked_commons.bike_embedding.clip_embedding_calculator import ClipEmbeddingCalculator
from biked_commons.resource_utils import resource_path


Using java as the Java binary


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv("../../resources/datasets/split_datasets/bike_bench.csv", index_col=0)
data = data.iloc[:10]

In [8]:
CHECKPOINT_SIZE = 100
def generate_embeddings(data, number_rendering_servers: int, server_init_timeout_seconds: int = 90):
    records = data.to_dict(orient="records")
    executor = ThreadPoolExecutor(max_workers=number_rendering_servers * 2)
    rendering_engine = RenderingEngine(number_rendering_servers=number_rendering_servers,
                                       server_init_timeout_seconds=server_init_timeout_seconds)
    future_results = []
    embedding_calculator = ClipEmbeddingCalculator()

    def render_record(clip_record: dict):
        rendering_result = rendering_engine.render_clip(clip_record)
        print("Rendering result received from server...")
        png_data = cairosvg.svg2png(rendering_result.image_bytes)
        image = PIL.Image.open(BytesIO(png_data))
        print("Image loaded...")
        # image_tensor = to_tensor(image)
        # augmented = get_augmented_views_gpu(image_tensor)
        embedding_tensor = embedding_calculator.embed_images(image)
        print("Embedding tensor obtained...")
        return embedding_tensor

    for record in records:
        future_results.append(executor.submit(render_record, record))

    count = 0
    numpy_result_array = np.ndarray(shape=(1, 512))
    for result in concurrent.futures.as_completed(future_results):
        latest_result = result.result().detach().numpy().reshape((1, 512))
        numpy_result_array = np.concat([numpy_result_array, latest_result])
        count += 1
        if count % CHECKPOINT_SIZE == 0:
            check_point_csv = f"./checkpoint_{count}.csv"
            pd.DataFrame(numpy_result_array).to_csv(check_point_csv)
            print(f"Check point reached, saved to csv {check_point_csv}")

    data_frame = pd.DataFrame(numpy_result_array)
    data_frame.to_csv("./clip_embeddings.csv")


In [ ]:
generate_embeddings(data, number_rendering_servers=5, server_init_timeout_seconds=180)

Starting BikeCAD server on port 8083...
Starting BikeCAD server on port 8080...
Starting BikeCAD server on port 8084...
BikeCAD server started on port 8084.
BikeCAD server started on port 8083.
BikeCAD server started on port 8080.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Rendering result received from server...
Rendering result received from server...
Rendering result received from server...
Rendering result received from server...
Rendering result received from server...
Rendering result received from server...
Rendering result received from server...
Rendering result received from server...
Rendering result received from server...
Rendering result received from server...
Image loaded...
Image loaded...


AttributeError: 'PngImageFile' object has no attribute 'shape'

Image loaded...
Image loaded...
Image loaded...
Image loaded...
Image loaded...
Image loaded...
Image loaded...
Image loaded...
